# Validation — `kappa-lora-top50-metamathqa`

**What this measures:** The repo already benchmarks tuner contributions with `method_comparison/MetaMathQA/run.py` against a published results corpus (the requester's notebook and merged PRs #3257/#3342 are exactly this pattern), so we run the harness on a new sibling config `experiments/lora/llama-3.2-3B-rank32-kappa-top50` that turns the default-off `condition_number_top_fraction: 0.5` ON; the target `num_trainable_params` (a field the harness itself writes) measures whether κ-selection actually restricts LoRA to ceil(56×0.5)=28 of the q/v modules — the "halves trainable params" half of the claim — while `test_accuracy` ≥ 0.3636 (published row 0.3836 − the repo's demonstrated ±0.02 parity band) guards the "without losing fit" half. Caveat carried from the analysis: the ±0.02 band was demonstrated for RNG-shift parity; a selection change alters which modules train, so a within-band single run should be confirmed with ≥2 seeds before being read as matching, and the adalora-row accuracy anchor should be re-anchored on `results/lora--llama-3.2-3B-rank32.json` when that file is read at run time.

**How I read the claim:** The PR implements κ-LoRA as a pure module-selection knob: rank target_modules matches by the condition number of the base weight and inject LoRA only into the top fraction. The verification notebook, following the repo's contribution pattern (config beside the rank32 siblings → run.py → JSON compared to published rows), trains one arm — LoRA r=32 on q/v with condition_number_top_fraction=0.5, 5000 steps on MetaMathQA → GSM8K — with 2 seeds. Support means: num_trainable_params ≤ 5,505,024 (the all-q worst case of a 28-of-56 selection; 4,587,520 expected at a balanced 14/14 q/v mix, vs 9,175,040 full LoRA) while test_accuracy stays within ±0.02 of the standard-LoRA published row (floor 0.3636 from the adalora anchor), with train_time ≤ 919.7 s as informative cost support. Caveats: this protocol's candidate pool is the 56 attention projections, not the paper's full model matrices, and because q_proj carries 1.5× v_proj's LoRA parameters the achieved fraction can legitimately land anywhere in 40–60% — the notebook must print the selected modules and their κ so the actual mix is visible; the −4.5% memory claim cannot resolve on this harness (<0.3% of peak).

- ⚠️ Candidate pool: the paper ranks all LoRA-candidate weight matrices; this protocol targets only q_proj+v_proj (56 of the model's 196 linears). 'Top 50%' here means 28 of 56 attention projections; a paper-faithful 7-type config (196 candidates → 98 kept) would span ~32–68% of parameters because shapes range 4096r–11264r.
- ⚠️ Comparator: the claim says 'vs standard LoRA' but the anchor row is ADALORA (SVD parameterization, 18,353,664 trainable — not the 9,175,040 LoRA r=32 formula; filename rank32 vs config r=8). Re-anchor the accuracy and time guards on results/lora--llama-3.2-3B-rank32.json from the 76-file corpus when the notebook runs.
- ⚠️ Module-count halving ≠ parameter halving under mixed shapes: q_proj carries 1.5× v_proj's LoRA params, so the κ-ranked 28 yield anywhere in 40–60% of full params; the clean 'halves' holds only if the selection's q/v mix is ≈14/14, which nothing in the mechanism guarantees.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`c31f85af7859`](https://github.com/mayorquinmachines/peft/commit/c31f85af78598dd42ab83c8a19a3eeb7dc0ed5a0)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "c31f85af78598dd42ab83c8a19a3eeb7dc0ed5a0"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa-top50` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa-top50/adapter_config.json`:

```json
{
  "peft_type": "LORA",
  "base_model_name_or_path": "meta-llama/Llama-3.2-3B",
  "r": 32,
  "lora_alpha": 8,
  "lora_dropout": 0.0,
  "target_modules": ["q_proj", "v_proj"],
  "condition_number_top_fraction": 0.5
}
```

In [ ]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa-top50/adapter_config.json")).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa-top50/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa-top50"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `temporary_results/lora--llama-3.2-3B-rank32-kappa-top50--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["temporary_results/lora--llama-3.2-3B-rank32-kappa-top50--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy", "train_time", "total_time", "accelerator_memory_max"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5505024,
        "baseline": 18353664
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.3636,
        "baseline": 0.38362395754359363
    },
    {
        "metric": "train_time",
        "direction": "<=",
        "threshold": null,
        "baseline": 1097.5522341161268
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": null,
        "baseline": 1339.9807203459786
    },
    {
        "metric": "accelerator_memory_max",
        "direction": "<=",
        "threshold": null,
        "baseline": 22796042240.0
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5505024 with `test_accuracy` >= 0.3636, `train_time` <= None, `total_time` <= None, `accelerator_memory_max` <= None holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
loop:
  max_iterations: 8
  fix_code: true

benchmarks:
  - name: kappa-lora-top50-metamathqa
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/lora/llama-3.2-3B-rank32-kappa-top50
        results_glob: method_comparison/MetaMathQA/temporary_results/lora--llama-3.2-3B-rank32-kappa-top50--*.json
        method: lora
        smoke:
          params_path: method_comparison/MetaMathQA/default_training_params.json
          overrides:
            max_steps: 8
            eval_steps: 4
            max_new_tokens: 16
      scorer: num_trainable_params
      metrics:
        - name: num_trainable_params
          direction: min
          # 5,505,024 = 28 kept modules x 196,608 (q_proj r=32 params) — the all-q WORST case of a correct
          # ceil(56*0.5)=28 kappa-selected pool; full coverage is 9,175,040 = 28*(6144+4096)*32, the 14/14
          # q/v mix lands exactly on 4,587,520 (50.0%), the all-v floor on 3,670,016 (40.0%). "Halves" holds
          # iff measured <= 5,505,024; a full-coverage injection (9,175,040) fails.
          threshold: 5505024
          role: target
        - name: test_accuracy
          direction: max
          # "without losing fit": published row test_accuracy 0.38362395754359363 minus the repo's own
          # demonstrated +/-0.02 equivalence band (supra r=32 verification notebook). Baseline clears it.
          threshold: 0.3636
          role: guardrail
        - name: train_time
          direction: min
          role: cost
        - name: total_time
          direction: min
          role: cost
        - name: accelerator_memory_max
          direction: min
          role: cost
    baseline:
      source: method_comparison/MetaMathQA/results/adalora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 18353664
        test_accuracy: 0.38362395754359363
        train_time: 1097.5522341161268
        total_time: 1339.9807203459786
        accelerator_memory_max: 22796042240.0
    compute:
      # GPU tier: the claim is a training-run outcome (5000-step fine-tune + GSM8K eval) on Llama-3.2-3B.
      # Budget: published row total_time 1340 s; notebook ~26 min on A100 for the 5000-step run; add ~10 min
      # for the gated Llama-3.2-3B download (~6.5 GB bf16) and GSM8K eval => 5400 s for ONE arm.
      tier: gpu
      timeout_s: 5400
    held_constant:
      - "base model meta-llama/Llama-3.2-3B fetched once with HF_TOKEN (gated repo, id pinned in adapter_config.json)"
      - "target_modules [q_proj, v_proj] across all 28 layers — the exact 56-module candidate pool (28 q 3072x3072, 28 v 3072x1024)"
      - "r=32, lora_alpha=8, lora_dropout=0.0 — rank, alpha and LoRA math untouched, only module selection changes"
      - "all default_training_params.json values (5000 steps, seed, lr, batching) — the published protocol, no training_params.json override"
      - "GSM8K test split and the harness generation/eval procedure unchanged"
      - "single A100-class GPU per the notebook regime"
    avoid:
      - "unpinned model revisions or live APIs — only the pinned Hub id from adapter_config.json"
      - "wall-clock timing on shared CPU — time metrics come from the harness run_info on a dedicated GPU"
      - "gating the paper's -4.5% memory claim — halving a 9.18M-param adapter moves <0.3% of the 22.8 GB peak, below harness resolution"
      - "comparing num_trainable_params to the adalora row's 18,353,664 (SVD parameterization) — the bound derives from the closed-form full-coverage count 9,175,040"
    provenance:
      num_trainable_params: "user_guidance"
      num_trainable_params_threshold: "inferred"
      test_accuracy: "user_guidance"
      test_accuracy_threshold: "protocol_doc:method_comparison/MetaMathQA/results/adalora--llama-3.2-3B-rank32.json"
      metrics_choice: "maintainer_comment:@mayorquinmachines"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      experiments: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56"
      compute: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56"
      held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md"
```